# Self-Disclosure Detection Pipeline

Runs annotation, training, and the experiment grid on a Colab GPU.

**Runtime > Change runtime type > T4 GPU** before starting.

## Two modes

`DRY_RUN = True` uses an innocuous public text dataset to verify the whole
pipeline end to end and measure throughput. Run this first. It touches no
sensitive data and needs no ethics approval, so it can be done while the
Secondary Data Checklist is still with the supervisor.

`DRY_RUN = False` uses the real corpus. **Do not switch this until the
checklist is signed.**

The point of the dry run is that annotating tens of thousands of posts takes
hours, and discovering a batch size problem or a session timeout at that point
is expensive. Better to find it now on data that does not matter.

## 1. Setup

In [ ]:
!nvidia-smi

# Set before torch initialises CUDA. Long prompts make allocation bursty,
# and expandable segments let the allocator reuse a fragmented pool rather
# than failing on a large request while free memory exists in pieces.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Report the GPU rather than asserting one. Annotation and training need it;
# rebuilding the corpus and running the agreement study do not, and those
# are worth being able to do on a CPU runtime when GPU quota is exhausted.
import torch
GPU = torch.cuda.is_available()
if GPU:
    free, total = torch.cuda.mem_get_info()
    print(f"{torch.cuda.get_device_name(0)}: "
          f"{free/1e9:.1f} GB free of {total/1e9:.1f} GB")
else:
    print("No GPU. CPU-only stages will run; annotation and training will skip.")


In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn pandas matplotlib tqdm

In [ ]:
# Pull the project code.
REPO = "https://github.com/hsnnaw/dissertation.git"
PROJECT = "/content/project"

import os, sys

# Absolute paths throughout, so re-running this cell after the %cd below
# behaves the same as running it fresh.
if os.path.isdir(PROJECT):
    !cd $PROJECT && git pull --ff-only
else:
    !git clone $REPO $PROJECT

%cd $PROJECT
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("ready")


In [ ]:
# Persist outputs to Drive so a session timeout does not lose hours of work.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
WORK = Path("/content/drive/MyDrive/dissertation")
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / "outputs").mkdir(parents=True, exist_ok=True)
print(WORK)

## 2. Mode

The Secondary Data Checklist is signed, so this runs on the real corpus.
Set `DRY_RUN = True` to go back to the news-text rehearsal, which is worth
doing after any change to the annotation path.


In [ ]:
DRY_RUN = False

# Which stages to run. Both need a GPU. They default to off because both
# have already been done and their outputs are preserved: the 5,000 labels
# are on Drive, and the experiment metrics are committed to the repo and
# restored by the analysis cell.
#
# Turn RUN_ANNOTATION on to extend the corpus. Annotation appends and
# resumes on post_id, so it adds to what exists rather than redoing it.
# Turn RUN_TRAINING on to regenerate per-example predictions, which the
# per-community breakdown needs and which were lost with the runtime.
RUN_ANNOTATION = False
RUN_TRAINING = False

# Repeats the two headline conditions across three seeds and runs the
# class-weighting ablation. Both address the single-seed and missing-
# ablation gaps, and the repeats regenerate the per-example predictions
# the per-community breakdown needs. About two hours on a T4.
RUN_SEED_SWEEP = True

N_POSTS = 500 if DRY_RUN else 5_000

STRATEGY = "few_shot"          # best on the benchmark: 0.902 at 1.78s/post
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f"DRY_RUN={DRY_RUN}  posts={N_POSTS}  strategy={STRATEGY}")
print(f"annotation={'ON' if RUN_ANNOTATION else 'skipped'}  "
      f"training={'ON' if RUN_TRAINING else 'skipped'}  "
      f"seed sweep={'ON' if RUN_SEED_SWEEP else 'skipped'}")


## 3. Input data

The real corpus is the Low et al. Reddit Mental Health Dataset
([Zenodo 3941387](https://zenodo.org/records/3941387), ODC-PDDL). The full
record is 3.1 GB; the cells below take only the `post` timeframe for the
communities in use, roughly 800 MB, onto Colab's local disk rather than Drive.

In dry run mode this is skipped and a small corpus of neutral news text is
built instead, with the same shape: a `text` field, a `post_id`, and a
`subreddit`.


In [ ]:
# Low et al. Reddit Mental Health Dataset, Zenodo record 3941387.
# https://zenodo.org/records/3941387   ODC Public Domain Dedication & License
#
# Downloaded to Colab's local disk, not Drive. The full record is 3.1 GB and
# only a subset is needed.
import urllib.request
from pathlib import Path

RECORD = "3941387"
BASE = f"https://zenodo.org/records/{RECORD}/files"
RAW = Path("data/raw")

# One timeframe only. Mixing windows would make period a confound in the
# cross-community generalisation experiment. "post" is Jan-Apr 2020.
TIMEFRAME = "post"

# suicidewatch and EDAnonymous are deliberately excluded: crisis content and
# eating disorder content respectively. Thirteen support communities remain,
# which is ample. Record this exclusion and the reasoning in the write-up:
# a justified exclusion is a stronger position than an unexamined inclusion.
MH = ["addiction", "alcoholism", "adhd", "anxiety", "autism",
      "bipolarreddit", "bpd", "depression", "healthanxiety", "lonely",
      "ptsd", "schizophrenia", "socialanxiety", "mentalhealth"]

# Negative examples. Holding out whole communities is only meaningful if the
# held-out set spans both groups, so both are needed.
CONTROL = ["conspiracy", "divorce", "fitness", "guns", "jokes",
           "legaladvice", "meditation", "parenting", "personalfinance",
           "relationships", "teaching"]


def fetch(subreddit: str) -> Path:
    """Download one subreddit file, skipping it if already present."""
    name = f"{subreddit}_{TIMEFRAME}_features_tfidf_256.csv"
    dst = RAW / name
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    # Download to a temporary name and rename on success, so an interrupted
    # download is never mistaken for a complete file on the next run.
    part = dst.with_suffix(".part")
    urllib.request.urlretrieve(f"{BASE}/{name}?download=1", part)
    part.rename(dst)
    return dst


# Probe one small file before committing to the rest. These are named
# "features_tfidf_256", so confirm they carry post text and not only feature
# vectors. Two minutes here beats discovering it after 800 MB.
if DRY_RUN:
    print("dry run: skipping corpus download")
else:
    import pandas as pd

    RAW.mkdir(parents=True, exist_ok=True)
    probe = fetch("addiction")
    head = pd.read_csv(probe, nrows=3, low_memory=False)
    TEXT_CANDIDATES = ("post", "selftext", "body", "text")   # same as src.data
    found = next((c for c in TEXT_CANDIDATES if c in head.columns), None)

    print(f"{probe.name}: {len(head.columns)} columns")
    if found is None:
        raise SystemExit(
            f"No text column in {probe.name}. Looked for {TEXT_CANDIDATES}.\n"
            f"Columns: {list(head.columns)[:20]}\n"
            "Without post text the annotation approach does not apply here."
        )
    print(f"text column: {found!r}")
    print(f"sample: {str(head[found].iloc[0])[:200]}")


In [ ]:
# Only run this once the probe above reports a usable text column.
if DRY_RUN:
    print("dry run: skipping corpus download")
else:
    import time

    started = time.perf_counter()
    for sub in MH + CONTROL:
        try:
            path = fetch(sub)
            print(f"  {path.stat().st_size/1e6:7.1f} MB  {path.name}")
        except Exception as exc:
            print(f"  FAILED  {sub}: {exc}")

    files = sorted(RAW.glob("*.csv"))
    total = sum(f.stat().st_size for f in files)
    print(f"\n{len(files)} files, {total/1e6:.0f} MB, "
          f"{time.perf_counter()-started:.0f}s")

    leftover = list(RAW.glob("*.part"))
    if leftover:
        print(f"\nincomplete, rerun this cell: {[p.name for p in leftover]}")


In [ ]:
import json, random
from pathlib import Path

POSTS = Path("data/interim/posts.jsonl")
POSTS.parent.mkdir(parents=True, exist_ok=True)

if DRY_RUN:
    from datasets import load_dataset

    # AG News: ordinary news text, nothing sensitive. Any neutral corpus of
    # comparable length would do equally well.
    ds = load_dataset("fancyzhx/ag_news", split=f"train[:{N_POSTS}]")
    topics = ["world", "sports", "business", "scitech"]

    with POSTS.open("w") as f:
        for i, row in enumerate(ds):
            f.write(json.dumps({
                "post_id": f"dry{i:05d}",
                "text": row["text"],
                "subreddit": topics[row["label"]],
                "group": "non_mh",
            }) + "\n")
    print(f"dry run corpus: {N_POSTS} posts")

else:
    # Real corpus. Requires the signed checklist.
    #
    # Put the dataset CSVs in data/raw/ first. prepare() reads every CSV in
    # that directory, takes the subreddit and timeframe from each filename,
    # and probes for the text column, which the releases name differently.
    RAW = Path("data/raw")
    csvs = sorted(RAW.glob("*.csv")) if RAW.exists() else []
    if not csvs:
        raise FileNotFoundError(
            f"No CSVs in {RAW.resolve()}. Copy the dataset there before "
            f"running this cell, for example from Drive:\n"
            f"  !mkdir -p {RAW} && cp /content/drive/MyDrive/<your-folder>/*.csv {RAW}/"
        )
    print(f"{len(csvs)} CSVs in {RAW}")

    from src.data import prepare
    prepare(
        input_dir=RAW,
        output_path=POSTS,
        sample=N_POSTS,
    )

print(sum(1 for _ in POSTS.open()), "posts ready")


## 4. Annotation

Runs the annotator locally on the Colab GPU. No text leaves the machine,
which is the condition the ethics route depends on.

Llama 3.1 is licence-gated. Accept the licence on the model page and supply
a read token below, or the model download fails with a gated-repo error.
`Qwen/Qwen2.5-7B-Instruct` is the ungated fallback, and is what the dry run
used.

At the measured 1.485 s/post this is roughly four hours for 10,000 posts.
Annotation is resumable: if the session drops, rerun this cell and it picks
up from where it stopped rather than starting over.

**Watch the confidence spread on the first batch.** Every dry run record
came back at 0.95 or above. If that repeats on real data the confidence
field carries no information, and the agreement study's stratified sample,
which splits at 0.8, collapses into a single band.


In [ ]:
if RUN_ANNOTATION:
    from huggingface_hub import login
    login()  # paste a token with read access
else:
    print("skipped: annotation is off")


In [ ]:
import time

# The labels file lives on Drive, not on Colab's disk. It is the expensive
# artefact -- hours of GPU time -- and a runtime recycle wipes local disk
# without warning. It carries post_id and labels only, no post text, so
# Drive is an appropriate place for it.
LABELS = WORK / f"labels_{STRATEGY}.jsonl"
LABELS.parent.mkdir(parents=True, exist_ok=True)
print(f"labels -> {LABELS}")
if LABELS.exists():
    print(f"  {sum(1 for _ in LABELS.open())} records")

if not RUN_ANNOTATION:
    print("\nskipped: annotation is off. Set RUN_ANNOTATION = True to extend")
    print("the corpus; it resumes on post_id and adds to what is already there.")
elif not GPU:
    raise SystemExit("Annotation needs a GPU. Runtime > Change runtime type.")
else:
    from src.annotate import run

    # Prove the write path before the model loads. An unmounted Drive or a
    # full quota fails here in a second, rather than after three minutes of
    # model load and hours of annotation with nowhere to put the results.
    _probe = LABELS.parent / "_writetest.tmp"
    try:
        with _probe.open("w") as _f:
            _f.write("probe\n")
            _f.flush()
        assert _probe.stat().st_size > 0
        _probe.unlink()
        print(f"  write test passed on {LABELS.parent}")
    except Exception as _exc:
        raise SystemExit(f"Cannot write to {LABELS.parent}: {_exc}")

    started = time.perf_counter()
    stats = run(
        input_path=POSTS,
        output_path=LABELS,
        strategy=STRATEGY,
        backend_kind="transformers",
        model=MODEL_ID,
        batch_size=16,
    )
    elapsed = time.perf_counter() - started

    n = stats["ok"] + stats["failed"]
    if n == 0:
        print("\nNothing to annotate: every post in POSTS is already in LABELS.")
    else:
        # Separate the one-off cost from the per-post cost. The model loads
        # once whatever the corpus size, so extrapolating from wall clock
        # overstates a longer run.
        annotation_s = stats["total_latency_s"]
        setup_s = elapsed - annotation_s
        print(f"\n{n} posts in {elapsed/60:.1f} min wall clock")
        print(f"  model load : {setup_s/60:5.1f} min  (one-off)")
        print(f"  annotation : {annotation_s/60:5.1f} min  "
              f"= {annotation_s/n:.3f} s/post")
        print(f"parse failure rate: {stats.get('failure_rate', 0):.1%}")


**Read the extrapolation before continuing.** If 20,000 posts would take longer
than a Colab session allows, the options are a larger batch size, a smaller
corpus, or splitting the run across sessions. Annotation is resumable, so the
last of those works: rerun the same cell and it picks up where it stopped.

A parse failure rate above a few percent means the prompt needs attention
before committing to the full run.

## 5. Splits

In [ ]:
from src.splits import load_labelled, stratified_split, subreddit_split, summarise

# The annotation output carries labels keyed by post_id but not the post
# text, so join it back from POSTS. Training needs both.
records = load_labelled(LABELS, posts_path=POSTS)
print(f"{len(records)} labelled records")

for mode, splitter in [("random", stratified_split), ("subreddit", subreddit_split)]:
    splits = splitter(records)
    summarise(splits, "is_disclosure")
    outdir = Path(f"data/processed/splits_{mode}")
    outdir.mkdir(parents=True, exist_ok=True)
    for name, group in splits.items():
        with (outdir / f"{name}.jsonl").open("w") as f:
            for r in group:
                f.write(json.dumps(r) + "\n")
    print(f"wrote {outdir}\n")


## 6. Lexical baseline

The codebook argues that disclosure often has to be inferred, and that a
lexical method therefore fails on exactly the implicit cases this project
cares about. That claim runs through the whole design and nothing has tested
it. It also leaves the transformer result unanchored: 0.92 F1 means little
until something simpler has been tried on the same splits.

TF-IDF over word and character n-grams with logistic regression, reported
overall and split by directness. Runs on CPU in under a minute.


In [ ]:
# No GPU needed. The interesting number is the explicit/implicit gap: if the
# baseline holds up on named conditions and falls away on inferred ones, the
# claim motivating the approach has evidence. If it does not, that is worth
# knowing before the claim goes into writing.
for mode in ["random", "subreddit"]:
    splits = Path(f"data/processed/splits_{mode}")
    if not (splits / "train.jsonl").exists():
        print(f"skipped {mode}: splits not built")
        continue
    print(f"\n{'=' * 60}\n{mode} split\n{'=' * 60}")
    out = f"outputs/experiments/baseline_{mode}"
    !python -m scripts.baseline --splits {splits} --output {out}


**Check the class balance before training.** If the positive rate is below
about 5%, the classifier will struggle regardless of class weighting, and the
corpus sampling needs rethinking rather than the model.

In dry run mode expect a very low positive rate, since news text contains no
self-disclosure. That is the correct result: it confirms the annotator is not
firing on everything.

## 7. Training

In [ ]:
if not RUN_TRAINING:
    print("skipped: training is off. Metrics are restored by the analysis")
    print("cell. Set RUN_TRAINING = True to regenerate per-example")
    print("predictions, which the per-community breakdown needs.")
elif not GPU:
    raise SystemExit("Training needs a GPU. Runtime > Change runtime type.")
else:
    from src.train import train

    result = train(
        splits_dir=Path("data/processed/splits_random"),
        output_dir=Path("outputs/experiments/gen_seen"),
        model_name="roberta-base",
        epochs=3,
        batch_size=16,
    )


## 8. Experiment grid

Skip in dry run mode. The numbers would be meaningless and it costs an hour.

In [ ]:
if not RUN_TRAINING:
    print("skipped: training is off")
else:
    # Noise robustness
    for rate in [0.0, 0.05, 0.10, 0.20, 0.30]:
        train(
            splits_dir=Path("data/processed/splits_random"),
            output_dir=Path(f"outputs/experiments/noise_{rate}"),
            noise_rate=rate,
        )

    # Generalisation
    train(
        splits_dir=Path("data/processed/splits_subreddit"),
        output_dir=Path("outputs/experiments/gen_unseen"),
    )

    # Size against cost
    for m in ["roberta-base", "distilroberta-base"]:
        train(
            splits_dir=Path("data/processed/splits_random"),
            output_dir=Path(f"outputs/experiments/size_{m}"),
            model_name=m,
        )

    # Class weighting ablation
    train(
        splits_dir=Path("data/processed/splits_random"),
        output_dir=Path("outputs/experiments/no_weights"),
        class_weights=False,
    )


## 9. Seed sweep and ablation

Every result up to here is a single run at seed 42. The generalisation gap
is 0.008 F1, and one run per condition cannot say whether that is an effect
or the variance of the training procedure. Three seeds on the two headline
conditions gives a standard deviation to quote it against.

Also runs the class-weighting ablation, and regenerates the per-example
predictions the per-community breakdown needs.

**About two hours on a T4.** Save to Drive as soon as it finishes.


In [ ]:
if not RUN_SEED_SWEEP:
    print("skipped: seed sweep is off")
elif not GPU:
    raise SystemExit("The seed sweep needs a GPU. Runtime > Change runtime type.")
else:
    !python -m scripts.seed_sweep --seeds 42 43 44

    # The ablation the write-up is missing. Cheap, and named in the feedback.
    from src.train import train
    train(
        splits_dir=Path("data/processed/splits_random"),
        output_dir=Path("outputs/experiments/no_weights"),
        class_weights=False,
    )


## 10. Directness disaggregation and sensitivity

Objective 6 asks for performance broken down by directness. The annotator's
directness labels cannot carry that: on 85 posts both sides called
disclosures, the codebook's naming criterion gives 50 explicit against the
reviewer's 60 and the annotator's 14. The criterion itself is mechanical, so
it is applied to the post text directly and both slicings are reported.

Also checks whether the headline rests on the two communities that dominate
the corpus. Neither needs a GPU, but both need `test_predictions.jsonl`,
which only exists once a training run has been done in this session.


In [ ]:
# Both read the per-example predictions, so they are skipped rather than
# failing if no training run has produced them in this runtime.
for name, splits in [("gen_seen", "splits_random"),
                     ("gen_unseen", "splits_subreddit")]:
    preds = Path(f"outputs/experiments/{name}/test_predictions.jsonl")
    if not preds.exists():
        print(f"skipped {name}: {preds} not present.")
        print("  Set RUN_SEED_SWEEP = True to regenerate it.")
        continue
    print(f"\n{'=' * 64}\n{name}\n{'=' * 64}")
    out = f"outputs/directness_breakdown_{name}.json"
    !python -m scripts.directness_breakdown --splits data/processed/{splits} --predictions {preds} --out {out}

    print(f"\n--- sensitivity: does the result rest on the largest communities? ---")
    sout = f"outputs/sensitivity_{name}.json"
    !python -m scripts.sensitivity --predictions {preds} --out {sout}


## 11. Analysis

In [ ]:
from pathlib import Path

EXPERIMENTS = Path("outputs/experiments")

# The grid ran on 2026-08-29 and the runtime was reclaimed before the
# outputs reached Drive. The metrics survived in the logs and are committed
# to the repo, so restore them rather than spending three hours of GPU
# reproducing numbers that would not come out identical anyway.
if not list(EXPERIMENTS.glob("*/results.json")):
    !python -m scripts.restore_results --dir {EXPERIMENTS}

# Aggregate metrics: the results tables.
!python -m scripts.collect_results --dir {EXPERIMENTS}

# The per-example breakdowns need test_predictions.jsonl, which the restore
# cannot reproduce. They come back if the runs are repeated.
if (EXPERIMENTS / "gen_seen" / "test_predictions.jsonl").exists():
    !python -m scripts.analyse breakdown --dir {EXPERIMENTS}/gen_seen --plot
    !python -m scripts.analyse noise --dir {EXPERIMENTS}
    !python -m scripts.analyse cost  --dir {EXPERIMENTS} --llm-ms 1876
    !python -m scripts.analyse gap   --dir {EXPERIMENTS}
else:
    print("\nper-example predictions absent, so the breakdowns are skipped.")
    print("Set RUN_TRAINING = True to regenerate them.")


## 12. Save to Drive

In [ ]:
import shutil

# Colab sessions are wiped without warning. Copy anything expensive to Drive
# as soon as it exists, not at the end of the notebook.
#
# What is deliberately NOT copied:
#
#   splits_*      contain verbatim post text, needed for training. They
#                 regenerate from the labels in seconds, so persisting them
#                 buys nothing and puts post text on third-party storage.
#   model,        half a gigabyte per training run, ten runs in the grid,
#   checkpoints   and nothing downstream reads them.
#
# The labels file IS copied and is the one thing that must survive: it costs
# hours to regenerate and carries no post text, only post_id and labels.
SKIP = shutil.ignore_patterns("splits_*", "*_with_text.jsonl",
                              "review_sample.jsonl", "posts.jsonl",
                              "model", "checkpoints", "checkpoint-*")

for src in ["outputs", "data/processed"]:
    dst = WORK / src.replace("/", "_")
    if Path(src).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True, ignore=SKIP)
        print(f"{src} -> {dst}")

# Fail loudly if post text reached Drive anyway, rather than discovering it
# later. Checks for the text field the training splits carry.
leaked = [f for f in WORK.rglob("*.jsonl")
          if any('"text":' in line for line in f.open().readlines()[:5])]
if leaked:
    print("\nWARNING: files on Drive contain post text:")
    for f in leaked:
        print(f"  {f}")

used = sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file())
print(f"\n{used / 1e6:.1f} MB in {WORK}")


## 13. Agreement study

Everything measured so far is agreement with the model's labels. Nothing
yet says whether those labels are *right*. This section draws a stratified
sample for manual review, which is what turns "the classifier reproduces
the annotator" into a claim about detecting disclosure.

No GPU needed. The sample is stratified by the model's confidence and its
judgement, so both bands and both classes are represented, and the file
written out carries the text alone: no label, no confidence, no type. The
review has to be blind or it measures nothing.

Label it on your own machine, not here. `input()` in Colab is painful over
a couple of hundred posts, and the sample contains post text, which is
better kept off cloud storage.


In [ ]:
import json
from src.splits import load_labelled

N_REVIEW = 200   # ~20s each, so about an hour. Scoring works on a partial
                 # pass, so stopping early costs precision, not the study.

# The labels carry post_id but deliberately no text, so join it back from
# POSTS. Both are deterministic, so the ids line up exactly.
joined = load_labelled(LABELS, posts_path=POSTS)
MERGED = Path("data/processed/labels_with_text.jsonl")
with MERGED.open("w") as f:
    for record in joined:
        f.write(json.dumps(record) + "\n")
print(f"{len(joined)} records with text")

SAMPLE = Path("data/processed/review_sample.jsonl")
!python -m src.agreement sample --input {MERGED} --output {SAMPLE} --n {N_REVIEW}


In [ ]:
# Download the sample, then label it locally:
#
#   python -m src.agreement label \
#       --sample data/processed/review_sample.jsonl \
#       --output data/processed/manual_labels.jsonl
#
# It is resumable: q saves and quits, and rerunning picks up where you
# stopped. Then score it against the model's labels:
#
#   python -m src.agreement score \
#       --model <your labels file> \
#       --manual data/processed/manual_labels.jsonl
from google.colab import files
files.download(str(SAMPLE))
